### Part 1: Indexing


In [1]:
import csv
import re
import csv
import re
import numpy as np
import math

from collections import defaultdict
from array import array
from typing import Dict, List, Tuple
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

In [2]:
import nltk
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/adriasoria/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/adriasoria/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/adriasoria/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

#### Build inverted index


In [3]:
def build_terms_query(text: str) -> List[str]:
    """Tokenize and normalize text into a list of lowercase terms.
    - Keeps only alphanumeric characters
    - Splits on non-alphanumerics
    """
    stemmer = PorterStemmer()
    stop_words = set(stopwords.words("english"))
    if not text:
        return []
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return tokens



In [4]:
def build_terms_doc(text: str) -> List[str]:
    """Tokenize text into a list of terms.
    """
    if not text:
        return []
    tokens = word_tokenize(text)
    return tokens

In [5]:
def create_inverted_index_from_csv_tf_idf(csv_path: str) -> Tuple[Dict[str, List[List]] , Dict[str, str]]:
    """Create an inverted index from a CSV file using title, description, and brand.

    Returns:
        index: { term: [[doc_id, array('I', positions)], ...] }
        title_index: { doc_id: title }
    """

    index = defaultdict(list)
    tf = defaultdict(list)  #term frequencies of terms in documents (documents in the same order as in the main index)
    df = defaultdict(int)  #document frequencies of terms in the corpus
    title_index = defaultdict(str)
    idf = defaultdict(float)


    with open(csv_path, mode="r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        documentos = list(reader)
        num_documents = len(documentos)

        for row in documentos:
            doc_id = row.get("pid")
            if not doc_id:
                # Skip rows without a valid identifier
                continue

            title = (row.get("title") or "").strip()
            description = (row.get("description") or "").strip()
            brand = (row.get("brand") or "").strip()


            # Concatenate selected fields
            content = f"{title} {description} {brand}"

            terms = build_terms_doc(content)
            title_index[doc_id] = title

            # Build per-document postings with positions
            current_page_index: Dict[str, List] = {}
            for position, term in enumerate(terms):
                if term in current_page_index:
                    current_page_index[term][1].append(position)
                else:
                    current_page_index[term] = [doc_id, array("I", [position])]

            # === Calculate normalize for TF ===
            norm = 0
            for term, posting in current_page_index.items():
                norm += len(posting[1]) ** 2
            norm = math.sqrt(norm)

            #calculate the tf(dividing the term frequency by the above computed norm) and df weights
            for term, posting in current_page_index.items():
                # append the tf for current term (tf = term frequency in current doc/norm)
                tf[term].append(np.round(len(posting[1]) / norm, 4)) ## SEE formula (1) above
                #increment the document frequency of current term (number of documents containing the current term)
                df[term] += 1 # increment DF for current term

            # Merge into global index
            for term_page, posting_page in current_page_index.items():
                index[term_page].append(posting_page)

    for term in df:
        idf[term] = np.round(np.log(float(num_documents / df[term])), 4)

    return index, tf, df, idf, title_index



In [6]:
import time
# Build the inverted index from the dataset and print basic stats
csv_path = "../../data/productos_preprocesados.csv"
start_time = time.time()
index, tf, df, idf, title_index = create_inverted_index_from_csv_tf_idf(csv_path)
print("Total time to create the TD-IDF index: {} seconds" .format(np.round(time.time() - start_time, 2)))
print(f"Documents indexed: {len(title_index):,}")
print(f"Vocabulary size: {len(index):,}")

# Show a few sample terms' posting counts
sample_terms = ["york", "track", "pant", "cotton"]
for t in sample_terms:
    if t in index:
        print(f"'{t}': {len(index[t])} postings")
    else:
        print(f"'{t}': 0 postings")



Total time to create the TD-IDF index: 4.54 seconds
Documents indexed: 28,080
Vocabulary size: 6,171
'york': 4 postings
'track': 1790 postings
'pant': 1982 postings
'cotton': 8145 postings


In [7]:
import numpy as np
import collections
from collections import defaultdict
import numpy.linalg as la

def rank_documents_tfidf_strict(terms, index, idf, tf, title_index, top_k=10):
    """
    Rank documents using cosine similarity with TF-IDF weights.
    Only considers documents that contain ALL query terms.
    """

    # Obtener los documentos que contienen todos los términos
    doc_sets = []
    for term in terms:
        if term not in index:
            return [], []  # Si un término no existe, ningún documento los contiene todos
        docs_for_term = {posting[0] for posting in index[term]}
        doc_sets.append(docs_for_term)

    # Si no hay conjuntos válidos
    if not doc_sets:
        return [], []

    # Intersección de documentos
    candidate_docs = set.intersection(*doc_sets)
    if not candidate_docs:
        return [], []

    # Inicializar vectores
    doc_vectors = defaultdict(lambda: [0] * len(terms))
    query_vector = [0] * len(terms)

    # Calcular frecuencia de términos en la query
    query_terms_count = collections.Counter(terms)
    query_norm = la.norm(list(query_terms_count.values())) or 1.0

    # Calcular TF-IDF
    for term_index, term in enumerate(terms):
        if term not in index:
            continue

        query_vector[term_index] = (query_terms_count[term] / query_norm) * idf[term]

        for doc_index, (doc_id, positions) in enumerate(index[term]):
            if doc_id in candidate_docs:
                doc_vectors[doc_id][term_index] = tf[term][doc_index] * idf[term]

    # Si no hay documentos con valores válidos
    if not doc_vectors:
        return [], []

    # Calcular similitud coseno (producto punto)
    doc_scores = [[float(np.dot(curDocVec, query_vector)), doc]for doc, curDocVec in doc_vectors.items()]

    # Si todos los scores son 0
    if not doc_scores:
        return [], []

    # Ordenar de mayor a menor score
    doc_scores.sort(reverse=True)

    ranked_docs = [x[1] for x in doc_scores][:top_k]

    return ranked_docs, doc_scores


#### Ranking


In [8]:
def search_tf_idf(query, index, idf, tf, title_index, top_k = 10):
    """
    Busca documentos que contengan TODOS los términos de la query,
    los rankea según TF-IDF y devuelve los resultados ordenados.
    """
    # Preprocesar la query (usa la misma función que para los documentos)
    terms = build_terms_query(query)

    # Si no hay términos válidos, devolver vacío
    if not terms:
        return [], []

    # Rankear documentos (solo los que contienen TODOS los términos)
    ranked_docs, doc_scores = rank_documents_tfidf_strict(
        terms, index, idf, tf, title_index, top_k
    )

    return ranked_docs, doc_scores


#### Test queries


In [9]:
test_queries = [
    "cotton multicolor track pant",
    "women black track pant pockets",
    "men cotton blue pant",
    "side pocket cotton pant",
    "slim cotton black pant",
]

top_k = 10
for q in test_queries:
    print(f"\n🔎 Query: {q}")
    ranked_docs, doc_scores = search_tf_idf(q, index, idf, tf, title_index, top_k)

    if not doc_scores:
        print("   No results found.")
        continue

    # Mostrar resultados con ranking, título y puntuación
    for rank, (score, doc_id) in enumerate(doc_scores[:top_k], start=1):
        title = title_index.get(doc_id, "(no title)")
        print(f"{rank:>2}. [{score:.3f}] {title} (id={doc_id})")



🔎 Query: cotton multicolor track pant
 1. [3.251] solid men multicolor track pant (id=TKPFDHDHGWXCFXPC)
 2. [3.023] solid women multicolor track pant (id=TKPFDHDGEYWM99XG)
 3. [2.977] checker women multicolor track pant (id=TKPFWJF7GG7YQSXM)
 4. [2.024] solid women multicolor track pant (id=TKPFCZ9EZDPZR5AH)
 5. [2.024] solid women multicolor track pant (id=TKPFCZ9ESZZ7YWEF)
 6. [2.024] solid men multicolor track pant (id=TKPFCZ9EHFCY5Z4Y)
 7. [2.024] solid women multicolor track pant (id=TKPFCZ9EHCNAPKPU)
 8. [2.024] solid women multicolor track pant (id=TKPFCZ9EFK9DNWDA)
 9. [2.024] solid women multicolor track pant (id=TKPFCZ9EA7H5FYZH)
10. [1.991] color block women multicolor track pant (id=TKPFCZ9EGGYENTZS)

🔎 Query: women black track pant pockets
 1. [4.001] solid women black track pant (id=TKPFTE9A4RNC7HUV)
 2. [3.658] solid women black track pant (id=TKPFPBJCY6M4UJBU)
 3. [3.612] solid women black track pant (id=TKPEQX5JXNUQFRFD)
 4. [3.606] solid women black grey track pant (

### Part 2: Evaluation

#### Evaluation metrics


In [10]:
import numpy as np


def _topk_indices_desc(y_score: np.ndarray, k: int) -> np.ndarray:
    n = len(y_score)
    if k >= n:
        return np.argsort(y_score)[::-1]
    idx = np.argpartition(y_score, -k)[-k:]
    return idx[np.argsort(y_score[idx])[::-1]]


def precision_at_k(doc_score: np.ndarray, y_score: np.ndarray, k: int = 10) -> float:
    order = _topk_indices_desc(np.asarray(y_score), k)
    topk_labels = np.asarray(doc_score)[order[:k]]
    relevant = np.sum(topk_labels == 1)
    return float(relevant) / max(k, 1)


def recall_at_k(doc_score: np.ndarray, y_score: np.ndarray, k: int = 10) -> float:
    total_relevant = int(np.sum(np.asarray(doc_score) == 1))
    if total_relevant == 0:
        return 0.0
    order = _topk_indices_desc(np.asarray(y_score), k)
    topk_labels = np.asarray(doc_score)[order[:k]]
    relevant_at_k = int(np.sum(topk_labels == 1))
    return float(relevant_at_k) / total_relevant


def avg_precision_at_k(doc_score: np.ndarray, y_score: np.ndarray, k: int = 10) -> float:
    order = _topk_indices_desc(np.asarray(y_score), k)
    number_of_relevant = 0
    prec_at_i_list = []
    number_to_iterate = min(k, len(order))
    for i in range(number_to_iterate):
        if np.asarray(doc_score)[order[i]] == 1:
            number_of_relevant += 1
            prec_at_i = number_of_relevant / float(i + 1)
            prec_at_i_list.append(prec_at_i)
    if number_of_relevant == 0:
        return 0.0
    return float(np.sum(prec_at_i_list)) / number_of_relevant

def f1_at_k(doc_score: np.ndarray, y_score: np.ndarray, k: int = 10) -> float:
    p = precision_at_k(doc_score, y_score, k)
    r = recall_at_k(doc_score, y_score, k)
    if (p + r) == 0:
        return 0.0
    return 2.0 * p * r / (p + r)

def mean_average_precision(list_of_doc_scores, list_of_y_scores, k: int = 10) -> float:
    if len(list_of_doc_scores) == 0:
        return 0.0
    ap_scores = []
    for doc_score, y_score in zip(list_of_doc_scores, list_of_y_scores):
        ap = avg_precision_at_k(np.asarray(doc_score), np.asarray(y_score), k)
        ap_scores.append(ap)
    return float(np.mean(ap_scores))

def mean_reciprocal_rank(list_of_doc_scores, list_of_y_scores, k=10):
    if isinstance(list_of_doc_scores, np.ndarray): list_of_doc_scores = [list_of_doc_scores]
    if isinstance(list_of_y_scores, np.ndarray): list_of_y_scores = [list_of_y_scores]
    rr_scores = []
    for doc_score, y_score in zip(list_of_doc_scores, list_of_y_scores):
        order = _topk_indices_desc(np.asarray(y_score), k)
        topk_labels = np.asarray(doc_score)[order[:k]]
        ranks = np.where(topk_labels == 1)[0]
        rr_scores.append(1.0 / (ranks[0] + 1) if len(ranks) else 0.0)
    return float(np.mean(rr_scores))


def dcg_at_k(doc_score: np.ndarray, y_score: np.ndarray, k: int = 10) -> float:
    order = _topk_indices_desc(np.asarray(y_score), k)
    labels = np.asarray(doc_score)[order[:k]]
    return np.sum([
        (labels[i]) / np.log2(i + 2) for i in range(len(labels))
    ])

def ndcg_at_k(doc_score: np.ndarray, y_score: np.ndarray, k: int = 10) -> float:
    dcg = dcg_at_k(doc_score, y_score, k)
    ideal_labels = np.sort(np.asarray(doc_score))[::-1][:k]
    ideal_dcg = np.sum([
        (ideal_labels[i]) / np.log2(i + 2) for i in range(len(ideal_labels))
    ])
    if ideal_dcg == 0:
        return 0.0
    return float(dcg) / float(ideal_dcg)


#### Evaluation testing


In [11]:
results, scores = search_tf_idf("women full sleeve sweatshirt cotton", index, idf, tf, title_index, top_k=10)

print("Top 10 Search Results:")
print("{:<20s} | {:<40s} | {:>10s}".format("Product ID", "Title", "Score"))
print("-" * 75)
for pid in results:
    # Find the score for this product ID from 'scores'
    score = None
    for s, p in scores:
        if p == pid:
            score = s
            break
    title = title_index.get(pid, "")
    print("{:<20s} | {:<40s} | {:10.4f}".format(pid, title, score if score is not None else 0.0))

Top 10 Search Results:
Product ID           | Title                                    |      Score
---------------------------------------------------------------------------
SWSFWTNDJFCF72WU     | full sleev solid women sweatshirt        |     3.2735
SWSFWTND3EPMFCQD     | full sleev solid women sweatshirt        |     3.2735
SWSFN2XZYZMGHD9A     | full sleev solid women sweatshirt        |     2.7997
SWSFMJF98EY2FXBH     | full sleev solid women sweatshirt        |     2.7610
SWSF5R7F2ZYCZYKH     | full sleev graphic print women sweatshirt |     2.6711
SWSFWGQFVXJGSY4E     | full sleev solid women sweatshirt        |     2.6702
SWSFWGQFKZ5HPGHH     | full sleev solid women sweatshirt        |     2.6702
SWSFW6KB8GZHEC72     | full sleev color block women sweatshirt  |     2.6500
SWSF5R7FY4A6AYHB     | full sleev graphic print women sweatshirt |     2.4538
SWSFXF56KYWBGQMD     | full sleev solid women sweatshirt        |     2.4365


In [12]:
results, scores = search_tf_idf("men slim jeans blue", index, idf, tf, title_index, top_k=10)

print("Top 10 Search Results:")
print("{:<20s} | {:<40s} | {:>10s}".format("Product ID", "Title", "Score"))
print("-" * 75)
for pid in results:
    # Find the score for this product ID from 'scores'
    score = None
    for s, p in scores:
        if p == pid:
            score = s
            break
    title = title_index.get(pid, "")
    print("{:<20s} | {:<40s} | {:10.4f}".format(pid, title, score if score is not None else 0.0))

Top 10 Search Results:
Product ID           | Title                                    |      Score
---------------------------------------------------------------------------
JEAF2QF6HNZHDXVX     | slim men dark blue jean                  |     3.2957
JEAFRAQXEKGUPNUN     | slim men blue jean                       |     3.1726
JEAFVPFUA97ZETDB     | slim men blue jean                       |     3.1443
JEAFTGSGTYKZGAEZ     | slim men blue jean                       |     3.1443
JEAFSGSYEAFCYHEE     | slim men blue jean                       |     3.1443
JEAFSFTWF3FHGNB3     | slim men blue jean                       |     3.1443
JEAFS57EYDGY4EJM     | slim men blue jean                       |     3.1443
JEAFS2KDU3UMSA7U     | slim men blue jean                       |     3.1443
JEAFKWR3FX9NH2V6     | slim men blue jean                       |     3.1443
JEAFGJ9Y9QKCYA6J     | slim men blue jean                       |     3.1443


In [13]:
test_queries_2_b = [
    "women full sleeve sweatshirt cotton",
    "men slim jeans blue"
]
ground_truth = {
    "women full sleeve sweatshirt cotton": {
        'SWSFWTNDJFCF72WU': 1,
        'SWSFWTND3EPMFCQD': 1,
        'SWSFN2XZYZMGHD9A': 1,
        'SWSFMJF98EY2FXBH': 1,
        'SWSF5R7F2ZYCZYKH': 1,
        'SWSFWGQFVXJGSY4E': 1,
        'SWSFWGQFKZ5HPGHH': 1,
        'SWSFW6KB8GZHEC72': 1,
        'SWSF5R7FY4A6AYHB': 1,
        'SWSFXF56KYWBGQMD': 1
    },

    "men slim jeans blue": {
        'JEAF2QF6HNZHDXVX': 1,
        'JEAFRAQXEKGUPNUN': 0,
        'JEAFVPFUA97ZETDB': 1,
        'JEAFTGSGTYKZGAEZ': 0,
        'JEAFSGSYEAFCYHEE': 0,
        'JEAFSFTWF3FHGNB3': 1,
        'JEAFS57EYDGY4EJM': 1,
        'JEAFS2KDU3UMSA7U': 0,
        'TKPFC9EA7H5YFZH': 1,
        'JEAFKWR3FX9NH2V6': 0
    }
}

top_k = 10
evaluation_results = {}

for q in test_queries_2_b:
    ranked_docs, doc_scores = search_tf_idf(q, index, idf, tf, title_index, top_k)
    if not ranked_docs:
        print(f"\n🔍 Query: '{q}' → No results found.")
        continue

    doc_scores = doc_scores[:top_k]
    # Crear vectores de relevancia y puntuación para las funciones
    y_true = np.array([ground_truth[q].get(doc, 0) for doc in ranked_docs])
# Extraer solo las puntuaciones numéricas del listado [score, doc_id]
    y_score = np.array([score for score, _ in doc_scores])

    metrics = {
        "Precision@10": precision_at_k(y_true, y_score, top_k),
        "Recall@10": recall_at_k(y_true, y_score, top_k),
        "F1@10": f1_at_k(y_true, y_score, top_k),
        "AvgPrecision@10": avg_precision_at_k(y_true, y_score, top_k),
        "nDCG@10": ndcg_at_k(y_true, y_score, top_k),
        "MAP": mean_average_precision([y_true], [y_score], top_k),
        "MRR": mean_reciprocal_rank([y_true], [y_score], top_k)
    }

    evaluation_results[q] = metrics


import pandas as pd
df_eval = pd.DataFrame(evaluation_results).T.round(3)
display(df_eval)


,Precision@10,Recall@10,F1@10,AvgPrecision@10,nDCG@10,MAP,MRR
women full sleeve sweatshirt cotton,1.0,1.0,1.000,1.00,1.000,1.00,1.0
men slim jeans blue,0.4,1.0,0.571,0.54,0.772,0.54,1.0


In [14]:
# Ground truth basado en los 10 resultados observados
ground_truth = {
    "cotton multicolor track pant": {
        'TKPFDHDHGWXCFXPC': 1,
        'TKPFDHDGEVYM99XG': 1,
        'TKPFWJ7FG97YQXSM': 1,
        'TKPFC9EZDPR7SAH': 0,
        'TKPFC9EZ7ZYWFEF': 0,
        'TKPFC9EHFY5Z4Y': 0,
        'TKPFC9EHNPAKPUJ': 0,
        'TKPFC9EK9DWMDA': 0,
        'TKPFC9EA7H5YFZH': 0,
        'TKPFC9EGGYENTZS': 1
    },

    "women black track pant pockets": {
        'TKPFTE9A4RNC7HUV': 1,
        'TKPFBDJ6YM4UJBU': 1,
        'TKPEQX5JXNUQFRD': 1,
        'TKPFDHDGS3D3BGUD': 0,
        'TKPEQX5JHTGFMHGC': 0,
        'TKPFDHDGAYXB3C9J': 1,
        'TKPFDHDH4PZGWMT': 1,
        'TKPFDHG9HEGYDTAS': 1,
        'TKPFYTUEXCNQVK5S': 1,
        'TKPFRHAGFDBFHZ3J': 1
    },

    "men cotton blue pant": {
        'TKPFYURWE3GGHWDT': 1,
        'TKPFJQFWJB2SPEPP': 1,
        'TKPFZ3MFWF7YHHYP': 1,
        'THFF9ZC7PDFDZAMY': 0,
        'TKPEGZ7MR2MIY6XZ': 0,
        'TKPEGZ7HNICZAYHK': 0,
        'TKPEGZ7HN6GCTRNE': 0,
        'TKPEXAZ6QUVVSYHP': 0,
        'TKPFYSG5XMYQBK6E': 0,
        'TKPFGTHHB9NYC9Y': 1
    },

    "side pocket cotton pant": {
        'TKPFGK5CUGWRDPM9': 1,
        'TKPFGK5CC5ZRARHZ': 1,
        'TKPFGK5C2ZR9GWDKA': 1,
        'TKPFW645SHJV23VH': 0,
        'TKPFDHDNYACU53ZG': 0,
        'TKPFDHDGEVYM99XG': 0,
        'TKPFDHDGS3D3BGUD': 0,
        'TKPF7ZEYFHX22AGG': 1,
        'TKPF7ZEYEHMG9GAH': 1,
        'TKPF7ZEYDUMGYBF5': 0
    },

    "slim cotton black pant": {
        'TKPFK3W6WB0DZCGG': 0,
        'TKPFK3W6WGCUTHR': 0,
        'TROPFSP654YMHHBS': 1,
        'TROE3KVQYHZG2DV': 1,
        'TROE3KVQ0BR4EPUZ': 1,
        'TROFQZJBMVNGHQKX': 0,
        'TROE2HZRCHVZAYDY': 1,
        'TROE2GSZFZMAQ8ZE': 1,
        'TROE2DVF5YUXHZ2U': 1,
        'TROE23TVFDFVF5AD': 1
    }
}

top_k = 10
evaluation_results = {}

for q in test_queries:
    ranked_docs, doc_scores = search_tf_idf(q, index, idf, tf, title_index, top_k)
    if not ranked_docs:
        print(f"\n🔍 Query: '{q}' → No results found.")
        continue

    doc_scores = doc_scores[:top_k]
    # Crear vectores de relevancia y puntuación para las funciones
    y_true = np.array([ground_truth[q].get(doc, 0) for doc in ranked_docs])
# Extraer solo las puntuaciones numéricas del listado [score, doc_id]
    y_score = np.array([score for score, _ in doc_scores])

    metrics = {
        "Precision@10": precision_at_k(y_true, y_score, top_k),
        "Recall@10": recall_at_k(y_true, y_score, top_k),
        "F1@10": f1_at_k(y_true, y_score, top_k),
        "AvgPrecision@10": avg_precision_at_k(y_true, y_score, top_k),
        "nDCG@10": ndcg_at_k(y_true, y_score, top_k),
        "MAP": mean_average_precision([y_true], [y_score], top_k),
        "MRR": mean_reciprocal_rank([y_true], [y_score], top_k)
    }

    evaluation_results[q] = metrics


import pandas as pd
df_eval = pd.DataFrame(evaluation_results).T.round(3)
display(df_eval)



,Precision@10,Recall@10,F1@10,AvgPrecision@10,nDCG@10,MAP,MRR
cotton multicolor track pant,0.1,1.0,0.182,1.000,1.000,1.000,1.000
women black track pant pockets,0.2,1.0,0.333,0.611,0.798,0.611,1.000
men cotton blue pant,0.1,1.0,0.182,0.500,0.631,0.500,0.500
side pocket cotton pant,0.0,0.0,0.000,0.000,0.000,0.000,0.000
slim cotton black pant,0.1,1.0,0.182,0.125,0.315,0.125,0.125
